# Run the rApp — twin-in-the-loop optimisation on a Virtual RIC

This notebook is the run engine behind the **Run the rApp** artefact page. It executes the
DigitalTwinSim rApp prototypes exactly as described in *Twin-in-the-Loop Optimisation as an
rApp on a Virtual RIC* (full description, v4): the existing configuration is first evaluated
in the twin from the ingested counter dump, candidate changes are rehearsed in the twin, and
only changes that pass the twin gate (§9) are emitted as a configuration intent.

**Engines:** Sionna RT (ray-traced twin, one isotropic trace per tower; every configuration
scored in post with the TR 38.901 pattern) and Sionna SYS (PHYAbstraction link curve and the
slice environment). **Modes:** single tower or cluster (§12). **Functions:** capacity
(twin optimiser), slice / PRB (SYS-scored guarded acceptance), audit (MCS histogram fit,
§10), energy (anchored to the validated RANOpt relationship).

**How to run:** Runtime → Run all. Without a package it runs the built-in study case
(single tower B, capacity). To run your own case, download a run package from the
[artefact page](https://ysrao.github.io/twin5GNR-multicell-SF/rapp-run.html) and upload it
when asked — the package carries the sites, the existing radio configuration, the ingested
counters and the function selected there. The run writes **rapp_result.json**; load that
file back on the artefact page to see the changes and the before → after performance.

In [ ]:
# --- Setup: platform, packages, the rApp modules and the scene ----------
import os, sys, subprocess, urllib.request

def _need(pkg, imp=None):
    try:
        __import__(imp or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for pkg, imp in [("sionna-rt==2.2.0", "sionna"), ("sionna", None), ("trimesh", None), ("rtree", None)]:
    _need(pkg, imp)

try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU / Metal path)")
except Exception:
    print("torch not present (only needed for the SYS link curve)")

REPO_RAW = "https://raw.githubusercontent.com/ysrao/twin5GNR-multicell-SF/main/"
os.makedirs("rapp", exist_ok=True)
for f in ["twin.py", "search.py", "cluster.py", "mcs_audit.py", "env_sys.py", "policy.py"]:
    if not os.path.exists(f"rapp/{f}"):
        urllib.request.urlretrieve(REPO_RAW + "rapp/" + f, f"rapp/{f}")
sys.path.insert(0, os.path.abspath("rapp"))
if not os.path.exists("city_open.ply"):
    urllib.request.urlretrieve(REPO_RAW + "data/city_open.ply", "city_open.ply")
print("modules + scene ready:", os.path.getsize("city_open.ply"), "bytes of scene")

In [ ]:
# --- The run package: sites + existing config + counters + function ----
import json

STUDY_PACKAGE = {
  "mode": "single", "function": "capacity",
  "sites": [{"site_id": "sf-twin-tower-B", "label": "Tower B",
             "position_enu_m": [250.0, 250.0], "ground_z_m": -64.75,
             "height_m_agl": 30.0, "azimuths_deg": [90.0, 210.0, 330.0],
             "tilts_deg": [10.0, 0.97, 3.92], "power_offset_db": 0.0,
             "traffic_shares_current": [0.4, 0.35, 0.25],
             "eval_half_m": 150.0, "eval_step_m": 20.0}],
  "site_traffic_totals": {"sf-twin-tower-B": 1.0},
  "policy": {"traffic_shares_projected": [0.34, 0.30, 0.36],
             "min_coverage_delta_pts": 0.0, "max_edge_drop_db": 1.0},
  "counters": {"median_dl": 38.0, "median_ul": 11.0, "edge_dl": 6.5,
               "edge_ul": 2.0, "fairness": 0.41, "prb_util": 81.0,
               "signalling": 34.0, "load": 0.75, "low_hours": 6,
               "carriers_per_site": 2, "current_shares": [0.6, 0.2, 0.2]},
  "pm": {"site_id": "sf-twin-tower-B",
         "gnb_configured": {"olla_offset_db": 0.0, "mcs_cap": 24},
         "mcs_histograms_by_sector": json.loads('{"0": [0.0, 0.0, 0.0, 0.0, 0.0, 0.04395, 0.0, 0.0, 0.0, 0.0, 0.0219, 0.0, 0.06445, 0.0, 0.0, 0.0203, 0.17165, 0.0444, 0.2833, 0.0, 0.3042, 0.04585, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "1": [0.0, 0.0, 0.0, 0.0, 0.0, 0.02455, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.09245, 0.0, 0.0, 0.0466, 0.04535, 0.069, 0.21025, 0.0, 0.2549, 0.2569, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "2": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0381, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0777, 0.0, 0.0, 0.11125, 0.07655, 0.07875, 0.11735, 0.0, 0.2711, 0.194, 0.0352, 0.0, 0.0, 0.0, 0.0, 0.0]}')},
}

def load_package():
    if os.path.exists("rapp_run_package.json"):
        print("package: rapp_run_package.json (local file)")
        return json.load(open("rapp_run_package.json"))
    try:
        from google.colab import files
        up = files.upload()
        for name in up:
            if name.endswith(".json"):
                print("package: uploaded", name)
                return json.loads(up[name])
    except Exception:
        pass
    print("package: built-in study case (single tower B, capacity)")
    return STUDY_PACKAGE

PKG = load_package()
MODE = PKG.get("mode", "single")
FUNCTION = PKG.get("function", "capacity")
print("mode:", MODE, "| function:", FUNCTION, "| sites:", [s["site_id"] for s in PKG["sites"]])

In [ ]:
# --- Shared helpers -------------------------------------------------------
import numpy as np
import twin as twin_mod
from search import optimise
from cluster import ClusterScorer, optimise_cluster

PLY = "city_open.ply"
RESULT = {"mode": MODE, "function": FUNCTION, "changes": [], "notes": []}
_traces = {}

def trace_for(site, rx, tag):
    key = (site["site_id"], tag)
    if key not in _traces:
        tx = [site["position_enu_m"][0], site["position_enu_m"][1],
              site["ground_z_m"] + site["height_m_agl"]]
        _traces[key] = twin_mod.trace_tower(PLY, tx, rx)
        print(f"trace {site['site_id']} ({tag}): {_traces[key]['trace_s']:.1f}s, {len(rx)} receivers")
    return _traces[key]

def cfg_of(site):
    return {"azimuths": list(site["azimuths_deg"]), "tilts": list(site["tilts_deg"]),
            "power_offset_db": float(site.get("power_offset_db", 0.0))}

def fmt_k(k):
    return (f"coverage {k['coverage_pct']:.1f}%  median RSRP {k['median_rsrp']:.1f} dBm  "
            f"edge p5 {k['edge_rsrp_p5']:.1f}  median SINR {k['median_sinr']:.1f} dB  "
            f"weighted SE {k['capacity_weighted_se']:.2f} bit/s/Hz")

def describe_changes(before, after):
    out = []
    for sid in before:
        b, a = before[sid], after[sid]
        lbl = sid.replace("sf-twin-tower-", "Tower ")
        for s in range(3):
            if a["azimuths"][s] != b["azimuths"][s]:
                out.append(f"{lbl} sector {s+1}: azimuth {b['azimuths'][s]:.0f} -> {a['azimuths'][s]:.0f} deg")
            if a["tilts"][s] != b["tilts"][s]:
                out.append(f"{lbl} sector {s+1}: tilt {b['tilts'][s]:.2f} -> {a['tilts'][s]:.0f} deg")
        if a.get("power_offset_db", 0.0) != b.get("power_offset_db", 0.0):
            out.append(f"{lbl}: power offset {b.get('power_offset_db',0.0):+.0f} -> {a.get('power_offset_db',0.0):+.0f} dB")
    return out

def clean(x):
    return None if x is None or (isinstance(x, float) and not np.isfinite(x)) else round(float(x), 2)

In [ ]:
# --- Capacity: the twin optimiser (Sionna RT + SYS link curve) -----------
if FUNCTION in ("capacity", "all"):
    policy = PKG["policy"]
    if MODE == "single":
        site = PKG["sites"][0]
        tx = [site["position_enu_m"][0], site["position_enu_m"][1],
              site["ground_z_m"] + site["height_m_agl"]]
        rx = twin_mod.street_receivers(PLY, tx[0], tx[1],
                                       half=site.get("eval_half_m", 150.0),
                                       step=site.get("eval_step_m", 20.0))
        scorer = twin_mod.TwinScorer(trace_for(site, rx, "single"), tx[:2])
        base = cfg_of(site)
        w_cur = twin_mod.traffic_weights(scorer, base, site["traffic_shares_current"])
        w_proj = twin_mod.traffic_weights(scorer, base, policy["traffic_shares_projected"])
        base_proj = scorer.score(base, w_proj)
        best, _, best_proj, best_cur, hist = optimise(scorer, base, w_proj, w_cur, policy)
        improved = best != base
        chosen = best if improved else base
        chosen_proj = best_proj if improved else base_proj
        print("AS-BUILT  (projected traffic):", fmt_k(base_proj))
        print("OPTIMISED (projected traffic):", fmt_k(chosen_proj))
        print("provenance:", "twin_guarded_accept" if improved else "no_improvement_keep_as_built",
              f"| candidates evaluated: {len(hist)}")
        Rb = scorer.sector_rsrp(base); Ra = scorer.sector_rsrp(chosen)
        mb = np.nanmax(np.where(np.isnan(Rb), -np.inf, Rb), axis=1)
        ma = np.nanmax(np.where(np.isnan(Ra), -np.inf, Ra), axis=1)
        RESULT.update({
            "configs_before": {site["site_id"]: base}, "configs_after": {site["site_id"]: chosen},
            "kpis_before": base_proj, "kpis_after": chosen_proj,
            "changes": describe_changes({site["site_id"]: base}, {site["site_id"]: chosen}),
            "provenance": "twin_guarded_accept" if improved else "no_improvement_keep_as_built",
            "maps": {"x": [clean(v) for v in rx[:, 0]], "y": [clean(v) for v in rx[:, 1]],
                     "before": [clean(v) for v in mb], "after": [clean(v) for v in ma]}})
    else:
        sites = PKG["sites"]
        rx = twin_mod.street_receivers(PLY, 0.0, 0.0, half=480.0, step=40.0)
        traces = {s["site_id"]: trace_for(s, rx, "cluster") for s in sites}
        scorer = ClusterScorer(traces, sites)
        base_configs = {s["site_id"]: cfg_of(s) for s in sites}
        totals = PKG["site_traffic_totals"]
        cur = {s["site_id"]: s["traffic_shares_current"] for s in sites}
        proj = policy["traffic_shares_projected"]
        w_cur = scorer.traffic_weights(base_configs, totals, cur)
        w_proj = scorer.traffic_weights(base_configs, totals, proj)
        base_proj = scorer.score(base_configs, w_proj)
        best, _, best_proj, best_cur, hist = optimise_cluster(scorer, base_configs, w_proj, w_cur, policy)
        improved = best != base_configs
        chosen = best if improved else base_configs
        chosen_proj = best_proj if improved else base_proj
        print("AS-BUILT  (projected traffic):", fmt_k(base_proj))
        print("OPTIMISED (projected traffic):", fmt_k(chosen_proj))
        print("provenance:", "twin_guarded_accept_cluster" if improved else "no_improvement_keep_as_built",
              f"| candidates evaluated: {len(hist)}")
        Rb = scorer.joint_rsrp(base_configs); Ra = scorer.joint_rsrp(chosen)
        mb = np.nanmax(np.where(np.isnan(Rb), -np.inf, Rb), axis=1)
        ma = np.nanmax(np.where(np.isnan(Ra), -np.inf, Ra), axis=1)
        RESULT.update({
            "configs_before": base_configs, "configs_after": chosen,
            "kpis_before": base_proj, "kpis_after": chosen_proj,
            "changes": describe_changes(base_configs, chosen),
            "provenance": "twin_guarded_accept_cluster" if improved else "no_improvement_keep_as_built",
            "maps": {"x": [clean(v) for v in rx[:, 0]], "y": [clean(v) for v in rx[:, 1]],
                     "before": [clean(v) for v in mb], "after": [clean(v) for v in ma]}})
    for c in RESULT["changes"]:
        print(" change:", c)

In [ ]:
# --- Slice / PRB: existing reward from the counters, SYS-scored choice ---
if FUNCTION in ("slice", "all"):
    import policy as pol
    from env_sys import SysCellEnv
    c = PKG["counters"]
    kpi_now = np.array([c["median_dl"], c["median_ul"], c["edge_dl"], c["edge_ul"],
                        c["fairness"], c["prb_util"], c["signalling"]], dtype=float)
    r_now = pol.reward(kpi_now)
    print(f"EXISTING (from the ingested counters): reward {r_now:.2f} "
          f"| shares {[round(v,2) for v in c['current_shares']]}")
    env = SysCellEnv(n_users=30, n_slots=100, seed=11)
    rows = []
    for i, (name, shares, sched) in enumerate(pol.PROFILES):
        k = env.evaluate(i)
        rows.append({"name": name, "shares": list(shares), "scheduler": sched,
                     "kpis": [round(float(v), 2) for v in k], "reward": round(pol.reward(k), 2)})
        print(f"  {name:16s} shares {shares}  reward {rows[-1]['reward']:.2f}  "
              f"median DL {k[0]:.1f}  edge DL {k[2]:.1f}  fairness {k[4]:.2f}")
    cur_idx = int(np.argmin([sum(abs(np.array(sh) - np.array(c["current_shares"])))
                             for _, sh, _ in pol.PROFILES]))
    cur = rows[cur_idx]
    ok = [r for r in rows if r["reward"] > cur["reward"]
          and r["kpis"][2] >= 0.95 * cur["kpis"][2] and r["kpis"][4] >= cur["kpis"][4] - 0.02]
    chosen = max(ok, key=lambda r: r["reward"]) if ok else cur
    prov = "policy_guarded_accept" if chosen is not cur else "guarded_reject_keep_current_profile"
    print("CHOSEN:", chosen["name"], "| provenance:", prov)
    RESULT["slice"] = {"counters_reward": round(r_now, 2), "profiles": rows,
                       "current_profile": cur["name"], "chosen_profile": chosen["name"],
                       "provenance": prov}
    RESULT["changes"].append(
        f"Slice profile {cur['name']} -> {chosen['name']}: PRB shares "
        f"{[round(v,2) for v in cur['shares']]} -> {[round(v,2) for v in chosen['shares']]}, "
        f"scheduler {chosen['scheduler']}")

In [ ]:
# --- Audit: twin reference MCS vs the PM histogram (v4 section 10) -------
if FUNCTION in ("audit", "all"):
    import mcs_audit
    pm = PKG["pm"]
    site = next(s for s in PKG["sites"] if s["site_id"] == pm["site_id"])
    tx = [site["position_enu_m"][0], site["position_enu_m"][1],
          site["ground_z_m"] + site["height_m_agl"]]
    rx = twin_mod.street_receivers(PLY, tx[0], tx[1],
                                   half=site.get("eval_half_m", 150.0),
                                   step=site.get("eval_step_m", 20.0))
    scorer = twin_mod.TwinScorer(trace_for(site, rx, "audit"), tx[:2])
    sinr, serving, have = mcs_audit.receiver_sinr(scorer, cfg_of(site))
    cfgd = pm["gnb_configured"]
    sectors = []
    for s in range(3):
        m = have & (serving == s)
        hist = np.array(pm["mcs_histograms_by_sector"][str(s)], dtype=float)
        f = mcs_audit.audit_sector(scorer, sinr[m], hist,
                                   cfgd["olla_offset_db"], cfgd["mcs_cap"])
        sectors.append(f)
        rec = f["recommendation"]
        print(f"sector {s+1}: twin mean MCS {f['twin_mean_mcs']:.1f} | PM mean {f['pm_mean_mcs']:.1f} "
              f"| fitted effective offset {f['estimated_effective_offset_db']:+.2f} dB (L1 {f['histogram_l1_distance']:.2f}) "
              f"| correction {('+' + format(rec['olla_offset_adjustment_db'], '.2f') + ' dB') if rec else 'none'} "
              f"| twin-verified SE gain {100*f['se_gain_relative']:+.1f}%")
    RESULT["audit"] = {"site": site["site_id"], "sectors": sectors}
    RESULT["notes"].append(
        "Audit fit note: the histogram fit is sensitive to the trace realisation - on a "
        "sparse stand-in histogram like this one, fitted offsets can move by several dB "
        "between runs. Read the L1 distance as the fit's confidence (lower is better; a "
        "fit at the edge of the search grid is not trustworthy). The calibration of "
        "record is the controlled test, where the PM file is generated from the same "
        "twin state: a planted -3 dB offset was recovered exactly, and a healthy cell "
        "produced no action. The guardrails (1 dB action threshold, +/-4 dB bound, "
        ">=2% twin-verified SE gain) are what make acting on a fit safe.")
    print(RESULT["notes"][-1])

In [ ]:
# --- Energy: sleep-window plan from the counters (anchored, v4 sec. 11) --
if FUNCTION in ("energy", "all"):
    c = PKG["counters"]
    util = float(c.get("prb_util", 81.0)); low = float(c.get("low_hours", 6))
    carriers = int(c.get("carriers_per_site", 2))
    saving = min(12.0, max(2.0, 7.21 * (low / 6.0) * (1.0 if util <= 88 else 0.85)))
    plan = []
    for s in PKG["sites"]:
        plan.append(f"{s.get('label', s['site_id'])}: sleep {carriers - 1} of {carriers} carriers "
                    f"for {low:.0f} low-traffic hours/night, 1 carrier kept on; slice floors and "
                    f"the twin coverage guardrail apply to any sleep decision")
    RESULT["energy"] = {"saving_pct": round(saving, 1), "index_before": 100.0,
                        "index_after": round(100.0 - saving, 1), "plan": plan}
    RESULT["changes"].extend(plan)
    print(f"energy index 100 -> {100.0 - saving:.1f} (saving {saving:.1f}%: RANOpt-validated "
          f"+7.21% scaled by {low:.0f} low-traffic hours and utilisation {util:.0f}%)")

In [ ]:
# --- Save the result for the artefact page --------------------------------
with open("rapp_result.json", "w") as f:
    json.dump(RESULT, f, indent=1, default=float)
print("rapp_result.json written")
print("headline:", RESULT.get("provenance", RESULT.get("slice", {}).get("provenance", "")),
      "| changes:", len(RESULT["changes"]))
try:
    from google.colab import files
    files.download("rapp_result.json")
except Exception:
    print("download rapp_result.json from the file browser and load it on the artefact page")